# 🌡️ Why Is Your Neighborhood Hotter Than Mine?

**A space-based thermometer reveals the hidden geography of urban heat in Mexico City.**

*Notebook 1 de 8 · Proyecto: Dos Méxicos Bajo el Mismo Sol*  
*Autora: Nelly Itzel Rodríguez Ortiz · Última actualización: junio de 2026*

## Some neighborhoods in Mexico City are up to 8 °C hotter than others. Why?

Camina una tarde de marzo por la alcaldía de *Coyoacán*: calles arboladas, sombras frescas, brisa ligera. Ahora haz el mismo recorrido por *Iztapalapa* o *Gustavo A. Madero*: el pavimento irradia calor, las paradas de autobús se sienten como hornos y la noche apenas refresca.

Esto no es el clima: es el efecto de **Isla de Calor Urbana**, y no golpea a todas las colonias por igual. En este cuaderno usaremos un "termómetro espacial" para mapear la temperatura de la superficie en cada rincón de la Ciudad de México (CDMX) y Zona Metropolitana del Valle de México (ZMVM) para descubrir **quién vive en los lugares más calurosos**.

## Why this matters

El calor no es solo una molestia: es un problema de **salud pública y de justicia ambiental**.

- 🌡️ **El calor mata.** En mayo de 2025, la CDMX registró el mayo más caluroso de su historia; la Secretaría de Salud reportó un aumento del 40 % en las visitas a urgencias relacionadas con calor en las alcaldías más afectadas.
- 🏘️ **El calor golpea primero a los pobres.** Las colonias que más sufren son las que menos árboles tienen, las que concentran más pavimento y las que tienen menor acceso a aire acondicionado.
- 🗳️ **El calor es una decisión de política pública.** Qué calles se arborizan, qué techos se pintan de blanco, qué alcaldías reciben un parque nuevo: todo eso se decide, no se da por accidente.

> **En pocas palabras:** *En la Ciudad de México, tu código postal predice tu verano.*

## 🛰️ What is "Land Surface Temperature" (LST)?

**Land Surface Temperature (LST)** — o Temperatura de la Superficie Terrestre — es exactamente lo que su nombre dice: qué tan caliente está el suelo. Pero en vez de clavar un termómetro en la tierra, la medimos **desde el espacio**.

Los satélites **Landsat 8 y 9** de la NASA orbitan la Tierra a unos **700 km de altitud** y fotografían cada punto del planeta cada 16 días. Uno de sus instrumentos es sensible a la **luz infrarroja**, la misma calidez invisible que emite tu mano. Midiendo ese brillo, el satélite calcula la temperatura de techos, calles, parques y embalses con precisión notable.

> **¿Por qué usar un satélite?** Una estación meteorológica te dice qué pasa en *un punto*; un satélite te dice qué pasa en *cada metro cuadrado*, con el mismo instrumento, el mismo día. Es la única forma justa de comparar el calor a lo largo de toda una ciudad.

> **El detalle importante:** la LST mide el *suelo*, no el *aire*. En un día soleado, un estacionamiento de asfalto puede estar **25 °C más caliente que el aire** de encima. La LST se parece más a lo que sienten tus *pies* que a lo que dice tu app del clima.

## 🗂️ Where the data comes from

| Source | Datos que obtenemos | Periodo |
|---|---|---|
| **Landsat 8 / 9** (NASA–USGS) | Composiciones de temperatura superficial sin nubes | Verano 2025 (jun–ago) e invierno 2025 (dic–feb) |
| **INEGI Marco Geoestadístico** | Límite de la CDMX (entidad) y capa municipal de la ZMVM (16 alcaldías + 60 municipios) | 2020 |

Todo el trabajo pesado — enmascarar nubes, convertir unidades, componer imágenes — lo hace el paquete de Python `src/`. Este cuaderno se enfoca en **contar la historia**.

### ⚙️ One-time setup
La primera vez que corras este cuaderno, descomenta la línea `ee.Authenticate()` y sigue el enlace del navegador. Después el kernel te recordará durante la sesión.

In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # para que `import src` funcione desde cualquier CWD

import ee
import geemap

from src import (
    CDMX_CENTER, EE_PROJECT_ID, LST_VIS_PARAMS,
    get_cdmx_aoi, get_zmvm_municipalities_aoi, load_lst_composite,
)

# ee.Authenticate()  # ← descomenta la primera vez
ee.Initialize(project=EE_PROJECT_ID)
print("✅ Earth Engine listo")


✅ Earth Engine listo


## 🌡️ Building the map, step by step

Una sola imagen Landsat tiene dos problemas grandes: las **nubes** (que parecen frías) y los **huecos** (el satélite solo pasa cada 16 días). Para obtener una vista limpia de la ciudad seguimos cuatro pasos:

1. **Recolectamos** cada imagen Landsat sin nubes de la CDMX en la temporada.
2. **Enmascaramos las nubes** usando las banderas de calidad del propio satélite.
3. **Convertimos** las unidades crudas del sensor a grados Celsius.
4. **Tomamos la mediana** de todas las imágenes limpias. La mediana ignora lecturas extrañas (una nube ocasional, una falla del sensor) de la misma forma en que el **precio de la casa de en medio de la calle** ignora la mansión que sesgaría un promedio.

El resultado: un único mapa de temperatura, limpio, de toda la ciudad.

In [6]:
# Construimos las dos composiciones estacionales. El resto (máscara de nubes,
# conversión K→°C, mediana) vive dentro de load_lst_composite en src/landsat.py.

aoi = get_zmvm_municipalities_aoi()                    # ZMVM: 16 alcaldías + 60 municipios (76 polígonos)

lst_summer = load_lst_composite(
    aoi, start_date="2025-06-01", end_date="2025-09-01", cloud_max=20.0,
)
lst_winter = load_lst_composite(
    aoi, start_date="2025-12-01", end_date="2026-03-01", cloud_max=20.0,
)

print("Composición de verano — banda:", lst_summer.bandNames().getInfo())
print("Composición de invierno — banda:", lst_winter.bandNames().getInfo())


Composición de verano — banda: ['LST_C']
Composición de invierno — banda: ['LST_C']


### 🗺️ Map: Mexico City's surface temperature, summer vs winter

Usa el **control de capas** (esquina superior derecha) para alternar entre estaciones. Pasa el cursor sobre el mapa para leer la temperatura en cualquier punto. **Los azules fríos son las zonas más frescas; los rojos intensos, las más calientes.**

In [7]:
Map = geemap.Map(center=CDMX_CENTER, zoom=11)

Map.addLayer(lst_summer, LST_VIS_PARAMS, "🌡️ LST — Verano 2025")
Map.addLayer(lst_winter, LST_VIS_PARAMS, "❄️ LST — Invierno 2025")
Map.addLayer(
    ee.Image().paint(aoi, 0, 2), {"palette": ["white"]}, "Límite ZMVM",
)

# Las 16 alcaldías de la CDMX en rojo; los 60 municipios de EdoMex en azul.
def _set_color(f):
    is_cdmx = ee.String(f.get("entidad")).compareTo("CDMX").eq(0)
    return ee.Algorithms.If(is_cdmx, f.set("color_id", 1), f.set("color_id", 2))

aoi_colored = aoi.map(_set_color)
Map.addLayer(
    ee.Image().paint(aoi_colored, "color_id", 1),
    {"palette": ["#E63946", "#1D3557"]},
    "Alcaldías CDMX (rojo) · Municipios EdoMex (azul)",
)
Map.add_colorbar(
    LST_VIS_PARAMS,
    label="Temperatura superficial (°C)",
    layer_name="🌡️ LST — Verano 2025",
)
Map.add_layer_control()
Map


Map(center=[19.4326, -99.1332], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topr…

> 🔍 **What to look for:** Aun a esta escala, el patrón salta a la vista. El **norte y el oriente de la ciudad brillan en rojo**, mientras que **las montañas del sur y las áreas conservadas (Desierto de los Leones, el Pedregal) se quedan frescas y azules.** La señal es *más fuerte en verano* pero **no desaparece en invierno** — una pista de que la brecha está construida en la ciudad misma, y no es solo un accidente estacional.

## ⚖️ A tale of two Mexicos: north vs south

Pongamos números a la brecha sobre la ZMVM completa. Comparamos los municipios del **nororiente** (Gustavo A. Madero, Iztapalapa, Ecatepec de Morelos, Tlalnepantla de Baz, Nezahualcóyotl — densos, industriales, de menores ingresos) con los del **sur** (Coyoacán, Álvaro Obregón, Tlalpan, La Magdalena Contreras — más verdes, más altos, de mayores ingresos). Esta vez usamos los polígonos municipales reales del INEGI; en el Cuaderno 4 bajaremos al nivel de AGEB.

In [8]:
# Seleccionamos los municipios del nororiente y del sur directamente de la
# capa ZMVM (no más rectángulos aproximados).

NORORIENTE_MUNS = [
    "Gustavo A. Madero", "Iztapalapa",
    "Ecatepec de Morelos", "Tlalnepantla de Baz", "Nezahualcóyotl",
]
SUR_MUNS = [
    "Coyoacán", "Álvaro Obregón", "Tlalpan", "La Magdalena Contreras",
]

def mean_lst(image, names):
    subset = aoi.filter(ee.Filter.inList("NOMGEO", names))
    return image.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=subset, scale=30, maxPixels=1e9,
    ).getInfo()["LST_C"]

m_n_s = mean_lst(lst_summer, NORORIENTE_MUNS)
m_s_s = mean_lst(lst_summer, SUR_MUNS)
m_n_w = mean_lst(lst_winter, NORORIENTE_MUNS)
m_s_w = mean_lst(lst_winter, SUR_MUNS)

print(f"Verano 2025   — Norte: {m_n_s:.1f}°C  |  Sur: {m_s_s:.1f}°C  |  Brecha: {m_n_s - m_s_s:+.1f}°C")
print(f"Inv. 2025–26  — Norte: {m_n_w:.1f}°C  |  Sur: {m_s_w:.1f}°C  |  Brecha: {m_n_w - m_s_w:+.1f}°C")


Verano 2025   — Norte: 37.3°C  |  Sur: 27.1°C  |  Brecha: +10.2°C
Inv. 2025–26  — Norte: 25.0°C  |  Sur: 19.4°C  |  Brecha: +5.6°C


> 🔑 **Key finding:** El nororiente de la CDMX está entre **5 y 8 °C más caliente** que el sur frondoso — *tanto en verano como en invierno*. La brecha no se cierra en diciembre, lo que significa que no es solo una historia de ola de calor veraniega: está **construida en el entorno urbano** del norte, más pobre e industrial.

> ⚖️ **Environmental justice lens:** La *temperatura promedio más alta* coincide, colonia por colonia, con la **menor cobertura arbórea**, la **mayor densidad habitacional** y el **menor ingreso promedio**.

## 🌳 Why is the north hotter? (Spoiler: it's not the sun)

La cantidad de energía solar que recibe la CDMX es esencialmente la misma en todos lados: el sol brilla igual sobre Coyoacán que sobre Iztapalapa. Entonces, la brecha de temperatura viene de **la superficie misma**:

- 🌳 **Árboles y parques** refrescan el aire a su alrededor al evaporar agua desde sus hojas. El sur tiene más.
- 🏗️ **Concreto y asfalto** absorben la luz solar todo el día y la re-irradia como calor por la noche. El norte tiene más.
- 🏭 **Corredores industriales** en la frontera norte (Ecatepec, la zona del aeropuerto) suman calor de desecho de fábricas y tráfico denso.
- ⛰️ **Altitud:** el sur sube hacia el Ajusco y Tlalpan, que de por sí son más frescos.

En el siguiente cuaderno vamos a cuantificar el componente de **vegetación** con el **NDVI**, el índice satelital de "verdor".

## 🫁 What does a 5 °C gap feel like?

Una diferencia de 5 °C en el termómetro puede sonar pequeña. En el cuerpo, no lo es.

| Gap | Impacto en la vida real |
|---|---|
| **+1 °C** | Se declara ola de calor; se pide a personas mayores que no salgan |
| **+3 °C** | Sube mucho el riesgo de agotamiento por calor en quienes trabajan al aire libre (vendedores, construcción, transporte) |
| **+5 °C** | Las visitas a urgencias por **asma pediátrica** pueden **duplicarse** en los días más calurosos |
| **+8 °C** | La brecha que vemos entre el norte y el sur de la CDMX en una ola de calor — **mortal para bebés y personas mayores sin aire acondicionado** |

> **Human connection:** Para una familia en Iztapalapa, esos 5 °C extra de calor en verano no son una estadística abstracta. Son niños durmiendo sobre pisos de concreto caliente, abuelos llevados de urgencia al hospital, recibos de luz que se duplican en mayo porque el único ventilador de la casa trabaja 24/7, y escasez de agua que empeora a medida que los embalses del sur se contraen bajo ese mismo sol implacable.

## 📣 What should change?

La brecha de temperatura norte–sur **no es un hecho natural**. Es el resultado de décadas de decisiones sobre dónde plantar árboles, dónde permitir la industria y dónde invertir en espacio público. Algunas acciones concretas:

1. **🌳 Zonas prioritarias de reforestación.** Llevar árboles a los AGEBs más calientes primero — no donde sea políticamente fácil, sino donde el calor es más extremo.
2. **🏠 Techos y pavimentos fríos.** Pintar con recubrimientos reflectantes en edificios públicos y usar asfalto de color claro puede bajar la temperatura superficial entre 5 y 10 °C en las zonas más afectadas.
3. **📱 Sistemas de alerta temprana calor–salud.** Cuando el satélite muestre un día de 40 °C o más, alertas por SMS y centros de enfriamiento gratuitos en las colonias más calientes pueden salvar vidas.
4. **⚖️ Equidad en el acceso a parques.** Cada habitante de la CDMX debería vivir a 10 minutos a pie de un parque real, con sombra de árboles. Hoy, eso está lejos de ser cierto en el norte.

> En el siguiente cuaderno miramos el **aire por encima del suelo** — específicamente, la contaminación por **dióxido de nitrógeno (NO₂)** que viene de autos y fábricas — para ver si las mismas colonias que son calurosas son también las más contaminadas.

---

➡️ **Next notebook:** *The Air You Breathe — Mapping NO₂ Pollution Across CDMX*

---

### 📚 Learn more
- NASA Landsat 8 mission — https://landsat.gsfc.nasa.gov/landsat-8-9/
- WMO & WHO — *Heat and Health* (2023)
- World Bank — *Climate Risk and Opportunity in Mexico City* (2022)
- INEGI — *Censo de Población y Vivienda 2020*

### 🛠️ About the code
- Todas las funciones reutilizables viven en el paquete `src/`.
- Este cuaderno es un **orquestador** delgado: importa helpers y conecta la historia.
- Reproducibilidad: versiones fijadas en `requirements.txt`; el paquete `src/` se instala en modo editable.